In [7]:
from train_policy import LAPAPolicy

model = LAPAPolicy("EleutherAI/pythia-160m")
model.eval()

print("loaded:", model.llm.config.name_or_path)
print("hidden_size:", model.llm.config.hidden_size)
print("vocab_size:", len(model.tokenizer))
print("act_token_start:", model.act_token_start)

/home/jianch2/miniconda3/envs/lapaTemp/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/jianch2/miniconda3/envs/lapaTemp/lib/python3.10/site-packages/transformers/modeling_utils.py:446: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are

TypeError: GPTNeoXForCausalLM.__init__() got an unexpected keyword argument 'dtype'

In [2]:
import sys, site
import ipykernel, zmq

print(sys.executable)
print("ENABLE_USER_SITE:", site.ENABLE_USER_SITE)
print(ipykernel.__file__)
print(zmq.__file__)

/home/jianch2/miniconda3/envs/lapaTemp/bin/python
ENABLE_USER_SITE: True
/home/jianch2/.local/lib/python3.10/site-packages/ipykernel/__init__.py
/home/jianch2/.local/lib/python3.10/site-packages/zmq/__init__.py


In [1]:
import os
os.environ["JAX_PLATFORMS"] = "cpu"

In [2]:
import jax
from flax.traverse_util import flatten_dict
from tux import JaxDistributedConfig, set_random_seed
from latent_pretraining.inference import LAPAInference
from latent_pretraining.delta_llama import VideoLLaMAConfig


/home/jianch2/miniconda3/envs/lapaTemp/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/home/jianch2/miniconda3/envs/lapaTemp/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.cloud.storage_control_v2 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.storage_control_v2 past that date.
  warnings.warn(message, FutureWarning)
/home/jianch2/miniconda3/envs/lapaTemp/lib

In [3]:
tokenizer = VideoLLaMAConfig.get_tokenizer_config()
llama = VideoLLaMAConfig.get_default_config()
tokenizer.vocab_file = "lapa_checkpoints/tokenizer.model"

JaxDistributedConfig.initialize(JaxDistributedConfig.get_default_config())
set_random_seed(1234)

CUDA backend failed to initialize: Found cuDNN version 8700, but JAX was built against version 8904, which is newer. The copy of cuDNN that is installed must be at least as new as the version against which JAX was built. (Set TF_CPP_MIN_LOG_LEVEL=0 and rerun for more info.)


In [ ]:
lapa = LAPAInference(
    image_size=256,
    tokens_per_delta=4,
    vqgan_checkpoint="lapa_checkpoints/vqgan",
    vocab_file="lapa_checkpoints/tokenizer.model",
    multi_image=1,
    jax_distributed=JaxDistributedConfig.get_default_config(),
    seed=1234,
    mesh_dim="1,-1,1,1",
    dtype="bf16",
    load_llama_config="7b",
    update_llama_config="dict(delta_vocab_size=8,sample_mode='text',theta=50000000,max_sequence_length=32768,scan_attention=False,scan_query_chunk_size=128,scan_key_chunk_size=128,scan_mlp=False,scan_mlp_chunk_size=8192,scan_layers=True)",
    load_checkpoint="params::lapa_checkpoints/params",
    tokenizer=tokenizer,
    llama=llama,
)

sampler = lapa.model


In [5]:

print("Model class:", type(sampler.model))
print("Config:")
print(sampler.config)

flat = flatten_dict(sampler.params, sep="/")
print("\nNumber of param leaves:", len(flat))

total = 0
for name, value in list(flat.items())[:80]:
    print(name, value.shape, value.dtype)
    total += value.size

print("\nFirst 80 leaves shown.")
print("Total params, approximate from all leaves:")
print(sum(v.size for v in flat.values()))

Model class: <class 'latent_pretraining.delta_llama.FlaxVideoLLaMAForCausalLM'>
Config:
VideoLLaMAConfig {
  "attn_pdrop": 0.0,
  "bos_token_id": 1,
  "delta_vocab_size": 8,
  "embd_pdrop": 0.0,
  "eos_token_id": 2,
  "fcm_max_ratio": 0.0,
  "fcm_min_ratio": 0.0,
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "max_sequence_length": 32768,
  "mesh_dim": "1,-1,1,1",
  "model_type": "video_llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "orig_sequence_length": 4096,
  "param_scan_axis": 0,
  "remat_attention": "",
  "remat_block": "",
  "remat_mlp": "",
  "resid_pdrop": 0.0,
  "rms_norm_eps": 1e-06,
  "sample_mode": "text",
  "scan_attention": false,
  "scan_key_chunk_size": 128,
  "scan_layers": true,
  "scan_mlp": false,
  "scan_mlp_chunk_size": 8192,
  "scan_query_chunk_size": 128,
  "theta": 50000000,
  "tie_vision_embeddings": false,
  "tie_word_embeddings": false,
  "transformers_version": "4.29.2",
  "use_cache": true,
  "use_f

In [11]:
lapa.model.params.items()

<generator object FrozenDict.items at 0x7f2150183530>

In [9]:
lapa.model.config

VideoLLaMAConfig {
  "attn_pdrop": 0.0,
  "bos_token_id": 1,
  "delta_vocab_size": 8,
  "embd_pdrop": 0.0,
  "eos_token_id": 2,
  "fcm_max_ratio": 0.0,
  "fcm_min_ratio": 0.0,
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "max_sequence_length": 32768,
  "mesh_dim": "1,-1,1,1",
  "model_type": "video_llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "orig_sequence_length": 4096,
  "param_scan_axis": 0,
  "remat_attention": "",
  "remat_block": "",
  "remat_mlp": "",
  "resid_pdrop": 0.0,
  "rms_norm_eps": 1e-06,
  "sample_mode": "text",
  "scan_attention": false,
  "scan_key_chunk_size": 128,
  "scan_layers": true,
  "scan_mlp": false,
  "scan_mlp_chunk_size": 8192,
  "scan_query_chunk_size": 128,
  "theta": 50000000,
  "tie_vision_embeddings": false,
  "tie_word_embeddings": false,
  "transformers_version": "4.29.2",
  "use_cache": true,
  "use_flash_attention": true,
  "vision_vocab_size": 8448,
  "vocab_size": 32000
}